# Pytorch

In [ ]:
import torch
import torch.nn as nn

print("PyTorch version:", torch.__version__)

## 1. Pytorch Basics

In [ ]:
# Create tensors
a = torch.tensor([1.0, 2.0, 3.0])
torch.manual_seed(87)                   # seed
b = torch.randn(2, 3)
print(a)
print(b)
print(a + b[0])
print(a + b[0, 0])      # broadcasting
print(a[0])
print(a[0].item())      # single-element tensor to a standard Python number 
print(a.tolist())       # tensor to a Python list

In [ ]:
# Tensor operations
torch.manual_seed(87)
c = torch.randn(2, 3) # 2X3
d = torch.randn(3, 2) # 3X2
print("Matrix multiplication c @ d:", c @ d) # 2X2

### 1.1 Autograd & Backpropagation

`autograd` is PyTorch's automatic differentiation engine. When a tensor has
`requires_grad=True`, PyTorch records every operation applied to it and builds a
**computational graph**. Calling `.backward()` then walks that graph
*backwards*, applying the **chain rule** at each node to accumulate gradients.

Consider a **composite function** $y = h(g(x))$ built from two steps:

$$u = g(x) = 3x + 1, \qquad y = h(u) = u^2$$

**Forward pass** (compute the value, left to right):

$$x = 2 \;\longrightarrow\; u = 3(2)+1 = 7 \;\longrightarrow\; y = 7^2 = 49$$

**Backward pass** (chain rule, multiply local derivatives from output back to input):

$$\frac{dy}{du} \;=\; 2u$$
$$\frac{dy}{dx} \;=\; \underbrace{\frac{dy}{du}}_{=\,2u}\cdot\underbrace{\frac{du}{dx}}_{=\,3} \;=\; (2\cdot 7)(3) \;=\; 42$$

Computational graph: `x → (·3, +1) → u → (^2) → y`. `backward()` starts from `y`
with $dy/dy = 1$ and propagates the gradient through each node in reverse, which is
exactly the chain rule. The result lands in `x.grad`.


In [ ]:
# Autograd on a composite function:  y = (3x + 1)^2
x = torch.tensor(2.0, requires_grad=True)   # input must be float/complex (int raises an error)

# Forward pass — PyTorch records each op in the computational graph
u = 3 * x + 1          # intermediate node:  u = g(x)
u.retain_grad()        # keep dy/du so we can inspect it (non-leaf grads are freed by default)
y = u ** 2             # output node:        y = h(u)

print("Forward:  x =", x.item(), "-> u =", u.item(), "-> y =", y.item())

# Backward pass — walk the graph in reverse, applying the chain rule
y.backward()

print("dy/du (local step)  :", u.grad.item())   # = 2u       = 14
print("dy/dx (full chain)  :", x.grad.item())   # = 2u * 3   = 42


In [ ]:
# The graph is DYNAMIC: it is rebuilt on every forward pass, so ordinary
# Python control flow (if / for / while) just works and is differentiated correctly.
x = torch.tensor([2.0], requires_grad=True)
if x > 1:
    y = x ** 2      # this branch is taken -> dy/dx = 2x = 4
else:
    y = x ** 3      # dy/dx = 3x^2

print(y)
y.backward()
print(x.grad)

In [ ]:
# DataFrame to tensor
import pandas as pd

df = pd.DataFrame({
    'x1': [1, 2, 3],
    'x2': [4, 5, 6]
})

df_tensor = torch.tensor(df.values, dtype=torch.float32)
print(df_tensor)

# numpy array to tensor
import numpy as np

arr = np.array([[1.2, 2.3, 3.5],
                [4.1, 0.2, 3.2]])

arr_tensor = torch.tensor(arr, dtype = torch.float32)
print(arr_tensor)

## 2. Build your model
### 2.1 Basic Ingredients
#### 2.1.1. Model structure: type of layer, dimension of layer, number of layers

In `pytorch`, there are many different ways to construct a neural network. Among them, we will check two methods.

The following two code blocks define the same neural network.

---

**Method A**
- custom forward pass is available
- possible to check intermediate output
- better for complex network

In [ ]:
class ModelMethodA(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 5)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(5, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

**Method B**
- easier to specify forward pass
- better for simple network

In [ ]:
class ModelMethodB(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [
            nn.Linear(10, 5),
            nn.ReLU(),
            nn.Linear(5, 1)
        ]
        self.net = nn.Sequential(*layers)   # * unpacks the list of layers

    def forward(self, x):
        return self.net(x)

---

In this notebook, we will stick to the method B.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[32, 64, 32], output_dim=1):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

In [ ]:
model = SimpleMLP(input_dim=10, hidden_dims = [16, 32, 16])
print(model)

#### 2.1.2. Loss function
- Regression: `torch.nn.MSELoss()`, `torch.nn.L1Loss()`, `torch.nn.HuberLoss()`, ...
- Classification: `torch.nn.CrossEntropyLoss()`, ...

In [ ]:
criterion = nn.MSELoss()

pred = torch.tensor([[1.0], [2.0], [3.0]])
true = torch.tensor([[0.1], [2.5], [2.8]])

loss = criterion(pred, true)                  # pred and target should be tensors

print("Loss:", loss.item())

#### 2.1.3. Optimizer
- GD & SGD: `torch.optim.SGD()`
- Adam: `torch.optim.Adam()`
- ...

In [ ]:
import torch.optim as optim

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.005)         # Core arguments: params of model, learning rate

### 2.2 Fitting, Evaluating, Validating

When you use your dataset, it is recommended to standardize (scale) the data beforehand.

In this example, all datasets are properly scaled.

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Data load
X_train, y_train = pd.read_csv("X_train.csv", header=None), pd.read_csv("y_train.csv", header=None)

# train validation split
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size = 0.2, random_state = 1)

In [ ]:
X_train.head()

In [ ]:
type(X_train)

In [ ]:
# Transform to tensor
X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_val = torch.tensor(X_val.values, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_val = torch.tensor(y_val.values, dtype=torch.float32)

In [ ]:
type(X_train)

Wrap the training tensors in a `DataLoader` so the example trains with **mini-batches**.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64
train_dataset = TensorDataset(X_train, y_train)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Validation / test stay as full tensors — no need to batch a forward-only pass
print(f"{len(train_dataset)} training samples -> {len(train_loader)} batches of up to {batch_size}")


In [ ]:
# Seed
torch.manual_seed(100)
np.random.seed(100)

# Instantiation
model = SimpleMLP(input_dim=X_train.shape[1], hidden_dims=[16, 32, 16], output_dim=1)

# Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)   # smaller lr suits frequent mini-batch updates

# Epochs (each epoch now performs several mini-batch updates)
epochs = 300

# History
train_losses = []
val_losses = []

# Best model
best_val_loss = float('inf')
best_model_state = None

print("Training started...")
print(f"{'='*50}\n")

for epoch in range(epochs):
    # ---- Training phase (mini-batch) ----
    model.train()                                # set the model to training mode
    running_loss = 0.0
    for X_batch, y_batch in train_loader:        # DataLoader yields mini-batches
        pred = model(X_batch)                    # (1) Forward
        loss = criterion(pred, y_batch)          # (2) Loss

        optimizer.zero_grad()                    # (3) Clear previous gradients
        loss.backward()                          #     Backprop (autograd)
        optimizer.step()                         # (4) Update parameters

        running_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)   # sample-weighted average
    train_losses.append(epoch_train_loss)

    # ---- Validation phase ----
    model.eval()                                 # set the model to evaluation mode
    with torch.no_grad():                        # disable gradient tracking
        val_pred = model(X_val)
        val_loss = criterion(val_pred, y_val)
        val_losses.append(val_loss.item())

        if val_loss.item() < best_val_loss:      # save best model
            best_val_loss = val_loss.item()
            best_model_state = model.state_dict()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}, Train Loss: {epoch_train_loss:.4f}, Val Loss: {val_loss.item():.4f}")

print(f"\n{'='*50}")
print("Training completed!")
print(f"Best val loss: {best_val_loss:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(train_losses, label='Training Loss', color='blue', alpha=0.7)
plt.plot(val_losses, label='Validation Loss', color='red', alpha=0.7)

plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Add annotation to highlight overfitting
if len(val_losses) > 50:
    # Find where validation loss starts increasing
    val_losses_array = np.array(val_losses)
    min_val_epoch = np.argmin(val_losses_array)
    plt.axvline(x=min_val_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Val Loss (Epoch {min_val_epoch})')
    plt.legend()

plt.tight_layout()
plt.show()

## 3. Potential Ways to Improve the Model
1. Network structure (deeper? wider?) and optimizer choice (`Adam`, ...)
2. Batch size & learning-rate tuning
3. Dropout (regularization)
4. ...


### 3.2 Batch size

Our training loop already uses a `DataLoader`, so **batch size** becomes a hyperparameter.
It controls how many samples the model sees before each weight update.

- **Small batches** (e.g. 16, 32): more updates per epoch and noisier gradients, which can
  act as a regularizer and help generalization — but each epoch is slower.
- **Large batches** (e.g. 256, or full-batch): smoother, more stable gradients and faster
  epochs, but fewer updates and sometimes weaker generalization.

A smaller batch usually pairs well with a smaller learning rate. Below we rebuild the
loader with a smaller batch size and retrain to compare.


In [ ]:
from torch.utils.data import DataLoader

# Try a smaller batch size (more, noisier updates per epoch)
small_batch_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
print(f"{len(small_batch_loader)} batches of up to 16")


In [ ]:
# Seed
torch.manual_seed(100)

# Instantiation
model = SimpleMLP(input_dim=X_train.shape[1], hidden_dims=[16, 32, 16], output_dim=1)

# Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.005)   # smaller lr for smaller/noisier batches

epochs = 300
train_losses = []
val_losses = []
best_val_loss = float('inf')
best_model_state = None

print("Training started...")
print(f"{'='*50}\n")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for X_batch, y_batch in small_batch_loader:
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = running_loss / len(small_batch_loader.dataset)
    train_losses.append(epoch_train_loss)

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val)
        val_loss = criterion(val_pred, y_val)
        val_losses.append(val_loss.item())
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_model_state = model.state_dict()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}, Train Loss: {epoch_train_loss:.4f}, Val Loss: {val_loss.item():.4f}")

print(f"\n{'='*50}")
print("Training completed!")
print(f"Best val loss: {best_val_loss:.4f}")


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(train_losses, label='Training Loss', color='blue', alpha=0.7)
plt.plot(val_losses, label='Validation Loss', color='red', alpha=0.7)

plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Add annotation to highlight overfitting
if len(val_losses) > 50:
    # Find where validation loss starts increasing
    val_losses_array = np.array(val_losses)
    min_val_epoch = np.argmin(val_losses_array)
    plt.axvline(x=min_val_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Val Loss (Epoch {min_val_epoch})')
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(best_model_state)             # Recover the best model
model.eval()

with torch.no_grad(): 
    y_test_pred = model(X_test)

test_loss = criterion(y_test_pred, y_test)

print(f'Test loss: {test_loss: .4f}')

### 3.3 Dropout
Dropout is a regularization technique to reduce overfitting in neural networks.

During training, it randomly **disables** (sets to zero) some neurons with a given probability `p`.

In [ ]:
class MLPwithDropout(nn.Module):
    def __init__(self, input_dim, hidden_dims=[32, 64, 32], output_dim=1, dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

In [ ]:
model = MLPwithDropout(input_dim=10, hidden_dims = [16, 32, 16], dropout=0.25)
print(model)

In [ ]:
# Seed
torch.manual_seed(100)

# Instantiation
model = MLPwithDropout(input_dim=X_train.shape[1], hidden_dims=[16, 32, 16], output_dim=1, dropout=0.25)

# Loss and Optimizer
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

epochs = 300
train_losses = []
val_losses = []
best_val_loss = float('inf')
best_model_state = None

print("Training started...")
print(f"{'='*50}\n")

for epoch in range(epochs):
    model.train()                                # dropout is ACTIVE in train mode
    running_loss = 0.0
    for X_batch, y_batch in train_loader:
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    model.eval()                                 # dropout is DISABLED in eval mode
    with torch.no_grad():
        val_pred = model(X_val)
        val_loss = criterion(val_pred, y_val)
        val_losses.append(val_loss.item())
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_model_state = model.state_dict()

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d}, Train Loss: {epoch_train_loss:.4f}, Val Loss: {val_loss.item():.4f}")

print(f"\n{'='*50}")
print("Training completed!")
print(f"Best val loss: {best_val_loss:.4f}")


In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(train_losses, label='Training Loss', color='blue', alpha=0.7)
plt.plot(val_losses, label='Validation Loss', color='red', alpha=0.7)
plt.title('Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Add annotation to highlight overfitting
if len(val_losses) > 50:
    # Find where validation loss starts increasing
    val_losses_array = np.array(val_losses)
    min_val_epoch = np.argmin(val_losses_array)
    plt.axvline(x=min_val_epoch, color='green', linestyle='--', alpha=0.7, label=f'Best Val Loss (Epoch {min_val_epoch})')
    plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(best_model_state)             # Recover the best model
model.eval()

with torch.no_grad(): 
    y_test_pred = model(X_test)

test_loss = criterion(y_test_pred, y_test)

print(f'Test loss: {test_loss: .4f}')

## 4. Exercise
The training set is available on the website. Give it a shot and see if you can beat my model!

```python
import pandas as pd

X_train, y_train = pd.read_csv("X_train.csv", header=None), pd.read_csv("y_train.csv", header=None)
```

**Rules**
1. We will use `nn.MSELoss()` to evaluate the performance on the test data.
2. No deeper than 20 layers.

The test set will be available at the end of the lab session.

### 4.1 Build your model with Codex

Use the provided Markdown file!

### 4.2 Evaluate with test set later.

```python
import torch.nn as nn

X_test, y_test = pd.read_csv("X_test.csv", header=None), pd.read_csv("y_test.csv", header=None)

model.load_state_dict(best_model_state)
model.eval()

with torch.no_grad(): 
    y_test_pred = model(X_test)

test_loss = nn.MSELoss(y_test_pred, y_test)

print(f'Test loss: {test_loss: .4f}')
````

In [ ]:
# import torch.nn as nn

# X_test, y_test = pd.read_csv("X_test.csv", header=None), pd.read_csv("y_test.csv", header=None)
# X_test, y_test = torch.tensor(X_test.values, dtype=torch.float32), torch.tensor(y_test.values, dtype=torch.float32)

# model.load_state_dict(best_model_state)
# model.eval()

# with torch.no_grad(): 
#     y_test_pred = model(X_test)

# test_loss = nn.MSELoss()(y_test_pred, y_test)

# print(f'Test loss: {test_loss: .4f}')